# Breast Cancer Prediction with pyTorch

TODO: check reproducibility

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Dataset

### Load

In [2]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [3]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

### Prepare

In [4]:
X_train_pt = torch.tensor(X_train, dtype=torch.float32)
X_test_pt = torch.tensor(X_test, dtype=torch.float32)
y_train_pt = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test_pt = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [5]:
print(X_train.shape, X_train_pt.shape)
print(y_train.shape, y_train_pt.shape)

(455, 30) torch.Size([455, 30])
(455,) torch.Size([455, 1])


In [6]:
print(X_test.shape, X_test_pt.shape)
print(y_test.shape, y_test_pt.shape)

(114, 30) torch.Size([114, 30])
(114,) torch.Size([114, 1])


In [7]:
train_dataset_pt = TensorDataset(X_train_pt, y_train_pt)

In [8]:
train_loader_pt = DataLoader(train_dataset_pt, batch_size=32, shuffle=True)

## Model

### Build (architecture)

In [9]:
class BCNet(nn.Module):  # breast cancer network

    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30, 64)  # fully connected (Dense in tf)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x))

        return x

In [10]:
model = BCNet()

### Compile

In [11]:
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()

### Training loop (fit)

In [12]:
epochs = 20

for epoch in range(epochs):

    model.train()  # training mode
    running_loss = 0.0

    for X_batch, y_batch in train_loader_pt:
        optimizer.zero_grad()  # reset all gradients

        preds = model(X_batch)
        loss = criterion(preds, y_batch)

        loss.backward()  # backpropagation
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}: Loss was {running_loss / len(train_loader_pt)}")  # average batch loss

Epoch 1: Loss was 0.6412562966346741
Epoch 2: Loss was 0.5200805564721426
Epoch 3: Loss was 0.38438575267791747
Epoch 4: Loss was 0.25908061464627585
Epoch 5: Loss was 0.1730548823873202
Epoch 6: Loss was 0.13256726264953614
Epoch 7: Loss was 0.12582177470127742
Epoch 8: Loss was 0.09207683367033799
Epoch 9: Loss was 0.0844506787136197
Epoch 10: Loss was 0.08243753152589003
Epoch 11: Loss was 0.08675488717854023
Epoch 12: Loss was 0.06707818284630776
Epoch 13: Loss was 0.08104619470735391
Epoch 14: Loss was 0.06255087579290072
Epoch 15: Loss was 0.056312942318618296
Epoch 16: Loss was 0.06019214609016975
Epoch 17: Loss was 0.05300052162880699
Epoch 18: Loss was 0.04109310523296396
Epoch 19: Loss was 0.06719045645246903
Epoch 20: Loss was 0.06045108279213309


## Results

### Predictions

In [13]:
# [TODO] code here

### Evaluation (accuracy)

In [14]:
with torch.no_grad():
    model.eval()  # evaluation mode

    preds = model(X_test_pt)
    loss = criterion(preds, y_test_pt).item()

    test_acc = ((preds >= 0.5) == y_test_pt).float().mean().item()


print(f"test_acc: {test_acc:.5%}")

test_acc: 98.24561%


### Show

In [15]:
# [TODO] code here

Plot wrongs